Note: Due to API quota limitations on the OpenAI account, the Hugging Face embedding model "sentence-transformers/all-MiniLM-L6-v2" was used to demonstrate the same embedding workflow.

Mentioned this change in readme file also. I want to transparent about this thats whyy.

In [21]:
from langchain_huggingface import HuggingFaceEmbeddings
import time
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded successfully.")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4212.96it/s]


Embedding model loaded successfully.


In [22]:

import os
print(os.getcwd())
print(os.listdir())

f:\TuteDude-Assignment\20-39\A22-Embedding-Models-Vector-Stores-and-Similarity-Search
['A22_Embedding_Models_VectorStores_Similarity_Search.ipynb', 'chroma_db', 'employees.csv', 'faiss_index', 'Logaa_Paramesh_Resume.pdf', 'notes.txt', 'pipeline_chroma', 'README.md', 'task6_ollama.py', '__pycache__']


In [23]:
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = []
documents.extend(TextLoader("notes.txt").load())
documents.extend(PyPDFLoader("Logaa_Paramesh_Resume.pdf").load())
documents.extend(CSVLoader("employees.csv").load())

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
document_chunks = text_splitter.split_documents(documents)

embedding = embedding_model.embed_query(document_chunks[0].page_content)

print("Embedding Vector Length:", len(embedding))

print("\nFirst 10 Values:")
print(embedding[:10])

Embedding Vector Length: 384

First 10 Values:
[-0.028264600783586502, -0.06470389664173126, 0.04130258783698082, 0.023597991093993187, 0.027609020471572876, 0.016336577013134956, -0.04025309160351753, -0.0010839015012606978, 0.026831354945898056, -0.0011767871910706162]


In [24]:
start_time = time.time()

all_embeddings = embedding_model.embed_documents([chunk.page_content for chunk in document_chunks])
end_time = time.time()

print("Total Embeddings Generated:", len(all_embeddings))
print("Embedding Dimension:", len(all_embeddings[0]))
print(f"Time Taken: {end_time - start_time:.2f} seconds")

Total Embeddings Generated: 18
Embedding Dimension: 384
Time Taken: 0.33 seconds


In [25]:
print("\nSample Embedding Values:")
print(all_embeddings[0][:10])

print("\nFirst Chunk Preview:")
print(document_chunks[0].page_content[:200])


Sample Embedding Values:
[-0.02826455794274807, -0.06470391899347305, 0.041302576661109924, 0.02359800785779953, 0.027609048411250114, 0.016336584463715553, -0.04025307670235634, -0.00108390545938164, 0.02683134935796261, -0.0011767708929255605]

First Chunk Preview:
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses n


### Observation

- Successfully generated embeddings for all document chunks using the Hugging Face model.
- Each chunk is represented as a 384-dimensional vector.
- These embeddings will be stored in a vector database for similarity search in the next tasks.

In [26]:
# Task 2 - Hugging Face Embedding Model

print("Model Name : sentence-transformers/all-MiniLM-L6-v2")
print("Embedding Dimension :", len(all_embeddings[0]))
print("Total Embeddings :", len(all_embeddings))

print("Sample Embedding Values:")
print(all_embeddings[0][:10])

Model Name : sentence-transformers/all-MiniLM-L6-v2
Embedding Dimension : 384
Total Embeddings : 18
Sample Embedding Values:
[-0.02826455794274807, -0.06470391899347305, 0.041302576661109924, 0.02359800785779953, 0.027609048411250114, 0.016336584463715553, -0.04025307670235634, -0.00108390545938164, 0.02683134935796261, -0.0011767708929255605]


#Task 3 – OpenAI vs Hugging Face Comparison
1. When should we prefer OpenAI Embeddings?

- Better semantic understanding.
- High-quality embeddings for production applications.
- Suitable for Retrieval-Augmented Generation (RAG) systems.
- Requires an internet connection and API credits.

2. When should we prefer Hugging Face Embeddings?

- Works offline after downloading the model.
- Free to use.
- Suitable for learning, local development and small projects.
- No API cost.

3. Cost vs Performance

| OpenAI | Hugging Face
|---------|--------------
| Paid API | Free
| Better semantic quality | Good semantic quality
| Cloud-based | Runs locally
| Internet required | Offline after download
| Best for production | Best for development and learning



In [27]:
# Task 4 - Document Similarity Search

from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(document_chunks, embedding_model)
print("FAISS Vector Store Created Successfully")

def search(query, top_k=3):
    results = vectorstore.similarity_search(query, k=top_k)
    print(f"\nQuery: {query}\n")
    for i, doc in enumerate(results, start=1):
        print(f"Result {i}:")
        print(doc.page_content[:250])
        print("-" * 50)

search("Artificial Intelligence")
search("Python")
search("Machine Learning")
search("Skills")
search("Experience")
search("Employee")


FAISS Vector Store Created Successfully

Query: Artificial Intelligence

Result 1:
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses neural networks with multiple layers to solve compl
--------------------------------------------------
Result 2:
LOGAA PARAMESH L T
 B.Tech — Computer Science (Specialization in Artificial Intelligence and Data Science)
 8870496955 | parameshlogaa@gmail.com | Bengaluru, India | linkedin.com/in/logaa-paramesh-l-t | github.com/Fearoff5214
CAREER OBJECTIVE
Motivat
--------------------------------------------------
Result 3:
hands-on expertise in Python, Java, SQL, and REST API development within a challenging and growth-oriented environment. Passionate
about building scalable, secure, and intelligent software systems — from backend services and cloud-integrated applicat
------------------------------------------

Observation is that the similarity search returned the most relevant document chunks for each query, FAISS compares embeddings instead of exact keywords, enabling semantic search

In [28]:
# Task 5 - Similarity Search using LangChain
query = "Artificial Intelligence"
results = vectorstore.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Document {i}")
    print(doc.page_content[:250])
    print("-" * 50)

Query: Artificial Intelligence

Document 1
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses neural networks with multiple layers to solve compl
--------------------------------------------------
Document 2
LOGAA PARAMESH L T
 B.Tech — Computer Science (Specialization in Artificial Intelligence and Data Science)
 8870496955 | parameshlogaa@gmail.com | Bengaluru, India | linkedin.com/in/logaa-paramesh-l-t | github.com/Fearoff5214
CAREER OBJECTIVE
Motivat
--------------------------------------------------
Document 3
hands-on expertise in Python, Java, SQL, and REST API development within a challenging and growth-oriented environment. Passionate
about building scalable, secure, and intelligent software systems — from backend services and cloud-integrated applicat
--------------------------------------------------
Document 4
anomalies, and pro

In [29]:
query = "Python"
results = vectorstore.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Document {i}")
    print(doc.page_content[:250])
    print("-" * 50)

query = "Machine Learning"
results = vectorstore.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Document {i}")
    print(doc.page_content[:250])
    print("-" * 50)

Query: Python

Document 1
practices applicable to application production support environments
AI-Based Intrusion Detection System · Python · Network Diagnostics · ML
– Developed a network intrusion detection pipeline in Python — designed modular data ingestion, preprocessing,
--------------------------------------------------
Document 2
Databases: MySQL, PostgreSQL, MongoDB — schema design, JOIN queries, indexing, query optimization; Oracle (familiar), SQL Server
(familiar)
Infrastructure & Platforms: Linux/UNIX (command-line proficient), Git, GitHub, Postman, VS Code; AWS basics (E
--------------------------------------------------
Document 3
hands-on expertise in Python, Java, SQL, and REST API development within a challenging and growth-oriented environment. Passionate
about building scalable, secure, and intelligent software systems — from backend services and cloud-integrated applicat
--------------------------------------------------
Document 4
anomalies, and produce structured a

observation is that the LangChain provides a simple abstraction for similarity search and Only the query is required to retrieve the most relevant document chunks.

about the task 6:
Ollama embeddings require a locally running Ollama server and cannot be executed in Google Colab


In [30]:
# Task 7 - FAISS Vector Store

vectorstore.save_local("faiss_index")
print("FAISS index saved successfully.")

from langchain_community.vectorstores import FAISS
loaded_vectorstore = FAISS.load_local("faiss_index",embedding_model,allow_dangerous_deserialization=True)
print("FAISS index loaded successfully.")

query = "Artificial Intelligence"

results = loaded_vectorstore.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print(doc.page_content[:250])
    print("-" * 50)

query = "Python"
results = loaded_vectorstore.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print(doc.page_content[:250])
    print("-" * 50)

FAISS index saved successfully.
FAISS index loaded successfully.
Query: Artificial Intelligence

Result 1
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses neural networks with multiple layers to solve compl
--------------------------------------------------
Result 2
LOGAA PARAMESH L T
 B.Tech — Computer Science (Specialization in Artificial Intelligence and Data Science)
 8870496955 | parameshlogaa@gmail.com | Bengaluru, India | linkedin.com/in/logaa-paramesh-l-t | github.com/Fearoff5214
CAREER OBJECTIVE
Motivat
--------------------------------------------------
Result 3
hands-on expertise in Python, Java, SQL, and REST API development within a challenging and growth-oriented environment. Passionate
about building scalable, secure, and intelligent software systems — from backend services and cloud-integrated applicat
---------------------

Observation is that the FAISS index was successfully saved to the local directory and after reloading the index, similarity search returned the expected document chunks.

In [31]:
# Task 8 - ChromaDB Vector Store
from langchain_community.vectorstores import Chroma

chroma_db = Chroma.from_documents(document_chunks,embedding_model,persist_directory="chroma_db")
print("ChromaDB created successfully.")

chroma_db.persist()
print("ChromaDB saved successfully.")

ChromaDB created successfully.
ChromaDB saved successfully.


In [32]:
query = "Artificial Intelligence"
results = chroma_db.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print(doc.page_content[:250])
    print("-" * 50)

Query: Artificial Intelligence

Result 1
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses neural networks with multiple layers to solve compl
--------------------------------------------------
Result 2
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses neural networks with multiple layers to solve compl
--------------------------------------------------
Result 3
LOGAA PARAMESH L T
 B.Tech — Computer Science (Specialization in Artificial Intelligence and Data Science)
 8870496955 | parameshlogaa@gmail.com | Bengaluru, India | linkedin.com/in/logaa-paramesh-l-t | github.com/Fearoff5214
CAREER OBJECTIVE
Motivat
--------------------------------------------------
Result 4
LOGAA PARAMESH L T
 B.Tech

In [33]:
loaded_chroma = Chroma(persist_directory="chroma_db",embedding_function=embedding_model)
print("ChromaDB loaded successfully.")

query = "Python"
results = loaded_chroma.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print(doc.page_content[:250])
    print("-" * 50)

ChromaDB loaded successfully.
Query: Python

Result 1
practices applicable to application production support environments
AI-Based Intrusion Detection System · Python · Network Diagnostics · ML
– Developed a network intrusion detection pipeline in Python — designed modular data ingestion, preprocessing,
--------------------------------------------------
Result 2
practices applicable to application production support environments
AI-Based Intrusion Detection System · Python · Network Diagnostics · ML
– Developed a network intrusion detection pipeline in Python — designed modular data ingestion, preprocessing,
--------------------------------------------------
Result 3
Databases: MySQL, PostgreSQL, MongoDB — schema design, JOIN queries, indexing, query optimization; Oracle (familiar), SQL Server
(familiar)
Infrastructure & Platforms: Linux/UNIX (command-line proficient), Git, GitHub, Postman, VS Code; AWS basics (E
--------------------------------------------------
Result 4
Databases: My

# Task 9 - FAISS vs ChromaDB Comparison
### 1. In-Memory vs Persistent Storage

**FAISS**
- Primarily stores vectors in memory.
- Index can be saved and loaded manually.

**ChromaDB**
- Stores vectors persistently on disk.
- Automatically manages the stored embeddings.

### 2. Use Cases for FAISS

- Fast similarity search
- Research and experimentation
- Local applications
- Large-scale vector indexing

### 3. Use Cases for ChromaDB

- Persistent document storage
- Retrieval-Augmented Generation (RAG)
- Semantic document search
- Production AI applications

In [34]:
# Task 10 - End-to-End Similarity Search Pipeline
def build_pipeline(embedding_backend="huggingface", vector_backend="faiss"):
    if embedding_backend == "huggingface":
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    if vector_backend == "faiss":
        vector_db = FAISS.from_documents(document_chunks, embeddings)

    elif vector_backend == "chroma":
      vector_db = Chroma.from_documents(document_chunks,embeddings,persist_directory="pipeline_chroma")

    return vector_db

pipeline = build_pipeline(embedding_backend="huggingface",vector_backend="faiss")
print("Pipeline created successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4782.93it/s]


Pipeline created successfully.


In [35]:
query = "Artificial Intelligence"
results = pipeline.similarity_search(query)
print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print(doc.page_content[:250])
    print("-" * 50)

pipeline = build_pipeline(embedding_backend="huggingface",vector_backend="chroma")
results = pipeline.similarity_search("Python")
print("Results using ChromaDB\n")
for doc in results:
    print(doc.page_content[:200])
    print("-" * 50)

Query: Artificial Intelligence

Result 1
Artificial Intelligence (AI) enables machines to perform tasks that normally require human intelligence.
Machine Learning is a subset of AI that allows systems to learn from data.
Deep Learning uses neural networks with multiple layers to solve compl
--------------------------------------------------
Result 2
LOGAA PARAMESH L T
 B.Tech — Computer Science (Specialization in Artificial Intelligence and Data Science)
 8870496955 | parameshlogaa@gmail.com | Bengaluru, India | linkedin.com/in/logaa-paramesh-l-t | github.com/Fearoff5214
CAREER OBJECTIVE
Motivat
--------------------------------------------------
Result 3
hands-on expertise in Python, Java, SQL, and REST API development within a challenging and growth-oriented environment. Passionate
about building scalable, secure, and intelligent software systems — from backend services and cloud-integrated applicat
--------------------------------------------------
Result 4
anomalies, and produce str

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4209.18it/s]


Results using ChromaDB

practices applicable to application production support environments
AI-Based Intrusion Detection System · Python · Network Diagnostics · ML
– Developed a network intrusion detection pipeline in Python
--------------------------------------------------
Databases: MySQL, PostgreSQL, MongoDB — schema design, JOIN queries, indexing, query optimization; Oracle (familiar), SQL Server
(familiar)
Infrastructure & Platforms: Linux/UNIX (command-line profici
--------------------------------------------------
Databases: MySQL, PostgreSQL, MongoDB — schema design, JOIN queries, indexing, query optimization; Oracle (familiar), SQL Server
(familiar)
Infrastructure & Platforms: Linux/UNIX (command-line profici
--------------------------------------------------
hands-on expertise in Python, Java, SQL, and REST API development within a challenging and growth-oriented environment. Passionate
about building scalable, secure, and intelligent software systems — f
-------------------

Observation is that the pipeline successfully converted document chunks into embeddings and Embeddings were stored in both FAISS and ChromaDB

# Task 11 - Observations & Insights
### 1. Importance of Embeddings in Generative AI

Embeddings convert text into numerical vectors while preserving semantic meaning. They help AI systems understand the relationship between words, sentences, and documents, enabling semantic search and document retrieval.

### 2. Why Vector Databases are Required

Vector databases store embeddings efficiently and perform fast similarity searches. They retrieve the most relevant documents based on semantic similarity instead of exact keyword matching.

### 3. How This Pipeline Enables RAG Systems

The pipeline loads documents, splits them into chunks, converts each chunk into embeddings, stores them in a vector database, and retrieves the most relevant chunks for a user query. These retrieved chunks can then be provided to a Large Language Model (LLM) to generate accurate and context-aware responses.



Right now, the build_pipeline() function only supports Hugging Face